In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scorecardpy as sc
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, roc_curve
import pickle

/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_excel('../data/default_credit_card.xls', header=1)
df.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default payment next month'],
      dtype='object')

In [3]:
pay_cols = ["PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"]
def status_map(x):
    if x<=0:
        return 'current'
    elif x==1:
        return 'M1' #1-30 days
    elif x==2:
        return 'M2' #31-60 days
    elif x==3:
        return 'M3' #61-90 days
    elif x == 4:
        return 'M4' #91-120 days(违约)
    elif x == 5:
        return 'M5' #121-150 days(违约)
    else:  # x >= 6
        return 'M6' #151+ days(严重违约)
for col in pay_cols:
    df[col+"_status"] = df[col].apply(status_map)

In [4]:
# 统计上月(PAY_2_status)到当月(PAY_0_status)的迁徙
transition_count = {}
states = ["current","M1","M2","M3","M4","M5","M6"]
trans_matrix = pd.DataFrame(0, index=states, columns=states)

for idx, row in df.iterrows():
    s_from = row["PAY_2_status"]
    s_to = row["PAY_0_status"]
    trans_matrix.loc[s_from, s_to] += 1
trans_matrix

,current,M1,M2,M3,M4,M5,M6
current,22735,1836,991,0,0,0,0
M1,0,28,0,0,0,0,0
M2,392,1672,1591,272,0,0,0
M3,47,109,71,41,58,0,0
M4,5,32,14,8,15,25,0
M5,3,7,0,1,3,0,11
M6,0,4,0,0,0,1,28


M2和M4的样本量较少(28和25)


In [5]:
# 转换为概率矩阵
trans_prob = trans_matrix.div(trans_matrix.sum(axis=1), axis=0)
trans_prob

,current,M1,M2,M3,M4,M5,M6
current,0.889406,0.071825,0.038768,0.000000,0.000000,0.000000,0.000000
M1,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
M2,0.099822,0.425770,0.405144,0.069264,0.000000,0.000000,0.000000
M3,0.144172,0.334356,0.217791,0.125767,0.177914,0.000000,0.000000
M4,0.050505,0.323232,0.141414,0.080808,0.151515,0.252525,0.000000
M5,0.120000,0.280000,0.000000,0.040000,0.120000,0.000000,0.440000
M6,0.000000,0.121212,0.000000,0.000000,0.000000,0.030303,0.848485


In [10]:
print("="*60)
print("迁徙矩阵分析")
print("="*60)

# 1. 样本量分析
print("\n1. 各状态样本量：")
for state in states:
    total = trans_matrix.loc[state].sum()
    print(f"   {state}: {total:,} 个账户")
print(f"   • M1状态只有28个样本，概率可能不可靠")
print(f"   • M5状态有25个样本，样本量也偏小")

#恶化率分析
print("\n2. 恶化率：")
state_rank = {state: i for i, state in enumerate(states)}
for state in states:
    if trans_matrix.loc[state].sum() > 0:
        worse = 0
        for to_state in states:
            if state_rank[to_state] > state_rank[state]:
                worse += trans_matrix.loc[state, to_state]
        worse_rate = worse / trans_matrix.loc[state].sum()
        if state != 'M6':
            print(f"   {state} → 恶化: {worse_rate:.2%}")
print(f"   • M5恶化率最高，其次是M4")


#改善率分析
print("\n3. 改善率：")
for state in states:
    if trans_matrix.loc[state].sum() > 0:
        better = 0
        for to_state in states:
            if state_rank[to_state] < state_rank[state]:
                better += trans_matrix.loc[state, to_state]
        better_rate = better / trans_matrix.loc[state].sum()
        if state != 'current':
            print(f"   {state} → 改善: {better_rate:.2%}")
print(f"   • M3改善率最高，其次是M4")

迁徙矩阵分析

1. 各状态样本量：
   current: 25,562 个账户
   M1: 28 个账户
   M2: 3,927 个账户
   M3: 326 个账户
   M4: 99 个账户
   M5: 25 个账户
   M6: 33 个账户
   • M1状态只有28个样本，概率可能不可靠
   • M5状态有25个样本，样本量也偏小

2. 恶化率：
   current → 恶化: 11.06%
   M1 → 恶化: 0.00%
   M2 → 恶化: 6.93%
   M3 → 恶化: 17.79%
   M4 → 恶化: 25.25%
   M5 → 恶化: 44.00%
   • M5恶化率最高，其次是M4

3. 改善率：
   M1 → 改善: 0.00%
   M2 → 改善: 52.56%
   M3 → 改善: 69.63%
   M4 → 改善: 59.60%
   M5 → 改善: 56.00%
   M6 → 改善: 15.15%
   • M3改善率最高，其次是M4


In [15]:
#对迁徙矩阵进行贝叶斯平滑
def smooth_transition_matrix(trans_matrix, alpha=1.0):
    states = trans_matrix.index.tolist()
    smooth_matrix = trans_matrix.copy()
    
    for state in states:
        total = trans_matrix.loc[state].sum()
        if total < 30:  # 若样本量小于30，则平滑
            global_avg = trans_matrix.sum(axis=0) / trans_matrix.sum().sum()
            for to_state in states:
                smooth_matrix.loc[state, to_state] = (
                    trans_matrix.loc[state, to_state] + alpha * global_avg[to_state]
                )
    return smooth_matrix

smooth_trans_matrix = smooth_transition_matrix(trans_matrix, alpha=2.0)
smooth_trans_prob = smooth_trans_matrix.div(smooth_trans_matrix.sum(axis=1), axis=0)
print("平滑后的迁徙概率矩阵")
print(smooth_trans_prob.round(4))

平滑后的迁徙概率矩阵
         current      M1      M2      M3      M4      M5      M6
current   0.8894  0.0718  0.0388  0.0000  0.0000  0.0000  0.0000
M1        0.0515  0.9415  0.0059  0.0007  0.0002  0.0001  0.0001
M2        0.0998  0.4258  0.4051  0.0693  0.0000  0.0000  0.0000
M3        0.1442  0.3344  0.2178  0.1258  0.1779  0.0000  0.0000
M4        0.0505  0.3232  0.1414  0.0808  0.1515  0.2525  0.0000
M5        0.1684  0.2684  0.0066  0.0378  0.1113  0.0001  0.4075
M6        0.0000  0.1212  0.0000  0.0000  0.0000  0.0303  0.8485


/var/folders/xr/kqyx90kx4yl3kskq3b0dnddh0000gn/T/ipykernel_29468/870520655.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.5454666666666668' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  smooth_matrix.loc[state, to_state] = (
/var/folders/xr/kqyx90kx4yl3kskq3b0dnddh0000gn/T/ipykernel_29468/870520655.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '28.245866666666668' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  smooth_matrix.loc[state, to_state] = (
/var/folders/xr/kqyx90kx4yl3kskq3b0dnddh0000gn/T/ipykernel_29468/870520655.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.1778' has dtype incompatible with int64, please explicitly cast to a 

In [20]:
#计算多期违约概率（PD）
def calculate_pd_smooth(trans_prob, periods=12):
    P = trans_prob.values
    P_periods = np.linalg.matrix_power(P, periods)
    default_states = ['M4', 'M5', 'M6']
    default_indices = [states.index(s) for s in default_states]
    
    pd_results = {}
    for i, state in enumerate(states):
        pd = sum(P_periods[i, j] for j in default_indices)
        pd_results[state] = min(pd, 1.0)
    return pd_results

pd_smooth = calculate_pd_smooth(smooth_trans_prob, periods=12)
print("PD:")
print({k: round(v, 3) for k, v in pd_smooth.items()})

PD:
{'current': 0.002, 'M1': 0.002, 'M2': 0.003, 'M3': 0.01, 'M4': 0.034, 'M5': 0.089, 'M6': 0.177}
